# Chapter 4 — Reasoning with probabilities · Notebook to complete

*Maths for AI, step by step*, Belk Siri — companion notebook.

[Open in Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/en/ch04_a_completer.ipynb) · Companion page: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/en.html

The cells to modify are marked **TO COMPLETE**. Run the cells in order: as long as an answer is missing, the checks say so without raising an error. The solution is in `ch04_corrige.ipynb`, on the same companion page.

**Contents.** Data · The calculations of lessons 4.1 to 4.5 · Exercises 4.1 to 4.6

**Before each calculation**, ask yourself which set you are counting in: spam messages, flagged messages or received messages.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colours of the book: one colour = one role
SPAM, LEGITIME, SIGNALE, DENOM = '#d62728', '#b9b6ae', '#1f4e8c', '#ffd54f'
BLEU, ORANGE = '#2a78d6', '#eb6834'   # “no spam missed”, normal distribution curve
from math import comb

## Data

The identifiers D4.1 and D4.2 are those of the book and of the CSV files. Variable and column names in the code are kept in French: `boite_20` and `boite_2` are the two mailboxes, `signalé` = flagged, `non signalé` = not flagged, `légitime` = legitimate, `durees` = processing times, `loi` = distribution, `manqué` = missed, `moyenne` = mean, `ecart_type` = standard deviation.

In [ ]:
# Chapter 4 data, copied from the book's CSV files (do not modify)
# Rows: spam, legitimate. Columns: flagged, not flagged.
LIGNES, COLONNES = ['spam', 'légitime'], ['signalé', 'non signalé']

# D4.1 — mailbox where 20 % of messages are spam (1 000 messages)
boite_20 = pd.DataFrame([[180, 20],
     [40, 760]],
                        index=LIGNES, columns=COLONNES)

# D4.2 — same filter, mailbox where 2 % of messages are spam (1 000 messages)
boite_2 = pd.DataFrame([[18, 2],
     [49, 931]],
                       index=LIGNES, columns=COLONNES)

# D4.3 — 1 000 processing times (ms), regenerated identically to the CSV file
durees = np.random.default_rng(2026).normal(40, 5, 1000).round(1)

## The calculations of the lessons, in Python

These cells redo the calculations of lessons 4.1 to 4.5.

In [ ]:
# Lesson 4.1 — count, then divide by the right group (equation 4.1)
t = boite_20
n = t.values.sum()
print('total:', n, '| spam:', t.loc['spam'].sum(), '| flagged:', t['signalé'].sum())
print('P(spam) =', t.loc['spam'].sum() / n, '| P(flagged) =', t['signalé'].sum() / n,
      '| P(spam and flagged) =', t.loc['spam', 'signalé'] / n)
# Among the flagged messages: the denominator is the “signalé” column
print(f"spam among flagged messages: {t.loc['spam', 'signalé']} / {t['signalé'].sum()} "
      f"≈ {t.loc['spam', 'signalé'] / t['signalé'].sum():.2f}")

In [ ]:
# Lesson 4.2 — the same filter in a mailbox where spam is rare (equation 4.2)
d, f = 0.90, 0.05                   # flagged shares: of spam, of legitimate messages
for p, t in [(0.20, boite_20), (0.02, boite_2)]:
    n_spam, n_leg = round(1000 * p), round(1000 * (1 - p))
    reconstruit = [[round(d * n_spam), n_spam - round(d * n_spam)], [round(f * n_leg), n_leg - round(f * n_leg)]]
    assert (np.array(reconstruit) == t.values).all()   # the table follows from the rates
    print(f'base rate {p:.0%}: {t.values.tolist()}  →  share of spam among flagged messages '
          f'= {d * p / (d * p + f * (1 - p)):.4f}')

In [ ]:
# Lesson 4.3 — two conditional probabilities, two denominators (equation 4.3)
def conditionnelle(t, case, groupe):
    """Share of the cell (row, column) within the group: a row ('spam') or a column ('signalé')."""
    denominateur = t.loc[groupe].sum() if groupe in t.index else t[groupe].sum()
    return t.loc[case] / denominateur, denominateur

for nom, t in [('20 %', boite_20), ('2 %', boite_2)]:
    rappel, den_r = conditionnelle(t, ('spam', 'signalé'), 'spam')
    precision, den_p = conditionnelle(t, ('spam', 'signalé'), 'signalé')
    print(f'mailbox with {nom}: P(flagged | spam) = {rappel:.2f} (denominator {den_r}); '
          f'P(spam | flagged) = {precision:.2f} (denominator {den_p})')

# The same numbers with scikit-learn, from message-by-message labels (if it is installed)
try:
    from sklearn.metrics import precision_score, recall_score
    vrai = [1] * 180 + [1] * 20 + [0] * 40 + [0] * 760      # 1 = spam
    predit = [1] * 180 + [0] * 20 + [1] * 40 + [0] * 760    # 1 = flagged
    print(f'scikit-learn: recall = {recall_score(vrai, predit):.2f}, precision = {precision_score(vrai, predit):.2f}')
except ImportError:
    print('scikit-learn not installed: this calculation is optional.')

In [ ]:
# Lesson 4.4 — Bayes' theorem (equations 4.4 and 4.5) and the book's curve
def bayes(p, d=0.90, f=0.05):
    p_signale = d * p + f * (1 - p)                  # equation 4.5
    return d * p / p_signale                        # equation 4.4

for p in (0.20, 0.02):
    print(f'P(spam) = {p}: P(flagged) = {0.9 * p + 0.05 * (1 - p):.3f}, P(spam | flagged) = {bayes(p):.4f}')
P = np.linspace(0.001, 0.5, 500)
plt.figure(figsize=(6, 3))
plt.plot(100 * P, bayes(P), color=SIGNALE, linewidth=2)
plt.scatter([2, 20], [bayes(0.02), bayes(0.2)], marker='s', color=SPAM, zorder=3)
plt.xlabel('Base rate: share of spam (%)'); plt.ylabel('P(spam | flagged)'); plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Lesson 4.5 — ten spam messages: binomial distribution (equation 4.7) and complementary event (equation 4.8)
n, p = 10, 0.1                        # we count over 10 SPAM messages; p = probability of being missed
loi = [comb(n, k) * p ** k * (1 - p) ** (n - k) for k in range(n + 1)]
print('0.9^10 =', round(0.9 ** 10, 4), '| at least one missed =', round(1 - 0.9 ** 10, 4))
print('P(X = k), k = 0 to 4:', [round(v, 3) for v in loi[:5]], '| sum =', round(sum(loi), 6))
couleurs = [BLEU] + [SPAM] * n
plt.figure(figsize=(6, 3))
plt.bar(range(n + 1), loi, color=couleurs, hatch=[''] + ['//'] * n, edgecolor='white')
plt.xlabel('Number of missed spam messages out of 10 spam messages'); plt.ylabel('Probability'); plt.show()

# A glimpse of the normal distribution: processing times (D4.3)
m, s = durees.mean(), durees.std()
print(f'mean = {m:.2f} ms, standard deviation = {s:.2f} ms; within ±1 standard deviation: {np.mean(abs(durees - m) <= s):.1%}, '
      f'within ±2 standard deviations: {np.mean(abs(durees - m) <= 2 * s):.1%}')

## Exercise 4.1 — Reading a chart: twenty spam messages

*Concepts: binomial distribution, complementary event (lesson 4.5).* First read the figure in the book.

### a) to d) Your readings

*Your answers:* …

In [ ]:
# TO COMPLETE: the binomial probability P(X = k) (equation 4.7)
def binomiale(n, p, k):
    return None   # TO COMPLETE; comb(n, k) gives the binomial coefficient

In [ ]:
# Check on the 20 spam messages of the exercise
if binomiale(20, 0.05, 0) is None:
    print('Complete the function.')
else:
    p0, p1 = binomiale(20, 0.05, 0), binomiale(20, 0.05, 1)
    print(f'P(X = 0) = {p0:.4f}, P(X = 1) = {p1:.4f}, at least one = {1 - p0:.4f} →',
          'correct' if abs(p0 - 0.95 ** 20) < 1e-9 and abs(p1 - 20 * 0.05 * 0.95 ** 19) < 1e-9 else 'check again')

## Exercise 4.2 — A more cautious filter

*Concepts: conditional probability (lesson 4.3), Bayes' theorem (lesson 4.4), base rate (lesson 4.2).* The cautious filter flags 75 % of spam messages and 1 % of legitimate messages. Mailbox of 10 000 messages, 10 % of which are spam. First do questions a) to e) on paper.

### a) to c) On paper

*Your answers:* …

### a) The table in Python

In [ ]:
# TO COMPLETE: the four counts of the table, for 10 000 messages
spam_signale = None
spam_non_signale = None
legitime_signale = None
legitime_non_signale = None

In [ ]:
# Check
cases = [spam_signale, spam_non_signale, legitime_signale, legitime_non_signale]
if None in cases:
    print('Complete the previous cell.')
else:
    prudent = pd.DataFrame([cases[:2], cases[2:]], index=LIGNES, columns=COLONNES)
    print(prudent)
    print('table:', 'correct' if cases == [750, 250, 90, 8910] else 'check again')

### b) The two conditional probabilities

In [ ]:
# TO COMPLETE: which group are you counting in? Write the quotient with the counts.
p_signale_si_spam = None
p_spam_si_signale = None

In [ ]:
# Check
for nom, la_votre, vrai in [('P(flagged | spam)', p_signale_si_spam, 0.75),
                            ('P(spam | flagged)', p_spam_si_signale, 750 / 840)]:
    print(f'{nom}:', 'not yet completed' if la_votre is None else
          ('correct' if abs(la_votre - vrai) < 0.001 else 'check again'))

### c) to e) Bayes' theorem as a function

In [ ]:
# TO COMPLETE: P(spam | flagged) from the base rate p and the filter rates d and f
def proba_spam_si_signale(p, d, f):
    return None   # TO COMPLETE (equations 4.4 and 4.5)

In [ ]:
# Check of c), d) and e)
if proba_spam_si_signale(0.1, 0.75, 0.01) is None:
    print('Complete the function.')
else:
    cas = [('c) cautious, 10 % spam', (0.10, 0.75, 0.01), 0.8929),
           ('d) cautious, 1 % spam', (0.01, 0.75, 0.01), 0.4310),
           ('e) cautious, 2 % spam', (0.02, 0.75, 0.01), 0.6048),
           ('e) lessons filter, 2 % spam', (0.02, 0.90, 0.05), 0.2687)]
    for nom, args, attendu in cas:
        v = proba_spam_si_signale(*args)
        print(f'{nom}: {v:.4f} →', 'correct' if abs(v - attendu) < 0.0001 else 'check again')
    P = np.linspace(0.001, 0.5, 500)
    plt.figure(figsize=(6, 3))
    plt.plot(100 * P, proba_spam_si_signale(P, 0.90, 0.05), color=SIGNALE, linewidth=2,
             label='lessons filter (90 %; 5 %)')
    plt.plot(100 * P, proba_spam_si_signale(P, 0.75, 0.01), color=SPAM, linewidth=2, linestyle='--',
             label='cautious filter (75 %; 1 %)')
    plt.axvline(2, color='black', linewidth=0.8, linestyle=':')
    plt.xlabel('Base rate: share of spam (%)'); plt.ylabel('P(spam | flagged)')
    plt.legend(); plt.grid(alpha=0.4); plt.show()

### d) and e) Your answers

*Your prediction for d), before calculating:* …

*Your choice in e):* …

## Exercise 4.3 — Three spam messages, then ten messages

*Concepts: Bernoulli trial and binomial distribution (lesson 4.5), conditional probability (lesson 4.3).* First do the questions on paper.

### a) to e) On paper

*Your answers:* …

In [ ]:
# TO COMPLETE: the probabilities of 0, 1, 2, 3 missed spam messages out of 3, then the answer to d)
loi_3 = None        # list of four numbers
p_d = None          # at least one spam gets through among 10 received messages

In [ ]:
# Check
if loi_3 is None or p_d is None:
    print('Complete the previous cell.')
else:
    print('b):', [round(v, 3) for v in loi_3], '→', 'correct' if np.allclose(loi_3, [0.729, 0.243, 0.027, 0.001]) else 'check again')
    print(f'd): {p_d:.4f} →', 'correct' if abs(p_d - 0.1829) < 0.0001 else 'check again')

## Exercise 4.4 — Simulating ten spam messages

*Concepts: binomial distribution (lesson 4.5), frequency and probability.* We count over **ten spam messages**.

In [ ]:
# TO COMPLETE: one series of 10 spam messages, each missed with probability 0.1
def simuler_serie(rng):
    """Returns the number of missed spam messages out of 10. rng.random() draws a number between 0 and 1."""
    return None   # TO COMPLETE

In [ ]:
# b) and c) 10 000 series (fixed seed to get the same numbers)
rng = np.random.default_rng(4)
if simuler_serie(rng) is None:
    print('Complete the function.')
else:
    rng = np.random.default_rng(4)
    X = np.array([simuler_serie(rng) for _ in range(10000)])
    freq = np.mean(X >= 1)
    print(f'observed frequency: {freq:.4f}; calculated probability: {1 - 0.9 ** 10:.4f} →',
          'correct' if abs(freq - (1 - 0.9 ** 10)) < 0.02 else 'check again')
    k = np.arange(6)
    plt.figure(figsize=(6, 3))
    plt.bar(k - 0.2, [np.mean(X == v) for v in k], width=0.4, color=SIGNALE, label='observed frequency')
    plt.bar(k + 0.2, [comb(10, v) * 0.1 ** v * 0.9 ** (10 - v) for v in k], width=0.4, color=SPAM,
            hatch='//', edgecolor='white', label='probability (equation 4.7)')
    plt.xlabel('Missed spam messages out of 10'); plt.legend(); plt.show()
    # d) More series: the frequency gets closer to the probability
    for N in (100, 100, 100, 100000):
        Y = np.array([simuler_serie(rng) for _ in range(N)])
        print(f'{N:>6} series: frequency = {np.mean(Y >= 1):.4f}')

### d) Your interpretation

*Your answer:* …

## Exercise 4.5 — Predict, then check: a spam campaign

*Concepts: independence, complementary event (lesson 4.5).*

### a) to c) Your predictions and calculations

*Your prediction:* …

In [ ]:
# TO COMPLETE: a campaign of 10 identical spam messages, all missed together with probability 0.1
def simuler_campagne(rng):
    return None   # TO COMPLETE: returns 10 or 0

In [ ]:
# d) Check by simulation
rng = np.random.default_rng(5)
if simuler_campagne(rng) is None:
    print('Complete the function.')
else:
    rng = np.random.default_rng(5)
    Z = np.array([simuler_campagne(rng) for _ in range(10000)])
    print(f'at least one missed: {np.mean(Z >= 1):.4f}; mean: {Z.mean():.3f} →',
          'correct' if abs(np.mean(Z >= 1) - 0.1) < 0.01 and abs(Z.mean() - 1) < 0.1 else 'check again')

## Exercise 4.6 — Challenge: how many spam messages before the risk of missing one exceeds 99 %?

*Concepts: Bernoulli and binomial distributions, normal distribution (lesson 4.5), standard deviation (lesson 1.1).*

### a) to d) Your proofs and calculations

*Your answers:* …

In [ ]:
# TO COMPLETE: b) the smallest n such that 0.9 ** n < 0.01; c) mean and standard deviation for n = 100
n_min = None
moyenne_100 = None
ecart_type_100 = None

In [ ]:
# Check, then d) binomial versus normal distribution
if None in (n_min, moyenne_100, ecart_type_100):
    print('Complete the previous cell.')
else:
    print(f'n_min = {n_min} →', 'correct' if n_min == 44 else 'check again')
    print(f'mean = {moyenne_100:.1f}, standard deviation = {ecart_type_100:.1f} →',
          'correct' if np.isclose(moyenne_100, 10) and np.isclose(ecart_type_100, 3) else 'check again')
    k = np.arange(0, 26)
    plt.figure(figsize=(6, 3))
    plt.bar(k, [comb(100, v) * 0.1 ** v * 0.9 ** (100 - v) for v in k], color=SPAM, alpha=0.8,
            label='binomial distribution (n = 100, p = 0.1)')
    xs = np.linspace(0, 25, 300)
    plt.plot(xs, np.exp(-(xs - 10) ** 2 / 18) / (3 * np.sqrt(2 * np.pi)), color=ORANGE, linewidth=2,
             label='normal distribution (mean 10, standard deviation 3)')
    plt.xlabel('Missed spam messages out of 100'); plt.legend(); plt.show()